In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
from pytorch_tabnet.tab_model import TabNetClassifier, TabNetRegressor
import torch
from imblearn.over_sampling import SMOTE

def print_feature_importance(importance, features, title):
    """Text-based feature importance display"""
    print(f"\n=== {title} ===")
    fi_df = pd.DataFrame({'Feature': features, 'Importance': importance})
    fi_df = fi_df.sort_values('Importance', ascending=False).head(15)
    print(fi_df.to_string(index=False))

# 1. Load and prepare data
file_path = r'C:\Users\sanga\OneDrive\Desktop\majorproject\talent_trajectory\EDA_Notebooks\datasets\cleaned_placement_dataset.csv'
data = pd.read_csv(file_path)



# Create interaction terms based on your columns
data['cgpa_x_internships'] = data['cgpa'] * data['internships']
data['dsa_x_projects'] = data['dsa'] * data['no_of_projects']

# Separate features and targets
X = data.drop(['is_placed', 'salary_as_fresher'], axis=1)
y_class = data['is_placed']
y_reg = data['salary_as_fresher']

# Identify numerical columns (excluding one-hot encoded)
num_cols = ['cgpa', 'inter_gpa', 'ssc_gpa', 'internships', 'no_of_projects', 
            'no_of_programming_languages', 'dsa', 'mobile_dev', 'web_dev', 
            'Machine Learning', 'cloud', 'cgpa_x_internships', 'dsa_x_projects']

# Scale only numerical features
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# 2. Split data
X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X, y_class, y_reg, test_size=0.15, random_state=42, stratify=y_class
)

# 3. Handle class imbalance
if y_class_train.mean() < 0.4 or y_class_train.mean() > 0.6:
    print("\nApplying SMOTE to handle class imbalance...")
    smote = SMOTE(random_state=42)
    X_train, y_class_train = smote.fit_resample(X_train, y_class_train)
    # For synthetic samples, set salary to median of placed students
    median_salary = y_reg_train[y_reg_train.notna()].median()
    y_reg_train = pd.Series(np.where(y_class_train == 1, median_salary, np.nan))

# 4. Placement Classifier
print("\nTraining Placement Classifier...")
clf = TabNetClassifier(
    optimizer_fn=torch.optim.Adam,
    optimizer_params={'lr': 2e-2, 'weight_decay': 1e-5},
    scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
    scheduler_params={'mode': 'max', 'patience': 5, 'factor': 0.5},
    mask_type='sparsemax',
    n_steps=5,
    n_d=32,
    n_a=32,
    gamma=1.3,
    lambda_sparse=1e-4,
    verbose=1
)

clf.fit(
    X_train.values, y_class_train.values,
    eval_set=[(X_test.values, y_class_test.values)],
    eval_metric=['accuracy', 'auc'],
    max_epochs=200,
    patience=30,
    batch_size=256,
    virtual_batch_size=128
)

# 5. Salary Regressor (only for placed students)
placed_train_mask = y_class_train == 1
if placed_train_mask.sum() > 0:
    print("\nTraining Salary Regressor...")
    reg = TabNetRegressor(
        optimizer_fn=torch.optim.Adam,
        optimizer_params={'lr': 2e-2, 'weight_decay': 1e-5},
        scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
        scheduler_params={'mode': 'min', 'patience': 5, 'factor': 0.5},
        n_steps=5,
        n_d=32,
        n_a=32,
        gamma=1.3,
        lambda_sparse=1e-4,
        verbose=1
    )

    reg.fit(
        X_train[placed_train_mask].values,
        y_reg_train[placed_train_mask].values.reshape(-1, 1),
        eval_set=[(X_test[y_class_test == 1].values, 
                 y_reg_test[y_class_test == 1].values.reshape(-1, 1))],
        eval_metric=['rmse', 'mae'],
        max_epochs=200,
        patience=30,
        batch_size=256,
        virtual_batch_size=128
    )

# 6. Evaluation
print("\n=== Model Evaluation ===")
y_class_pred = clf.predict(X_test.values)
print(f"Placement Accuracy: {accuracy_score(y_class_test, y_class_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_class_test, y_class_pred))

placed_test_mask = y_class_test == 1
if placed_test_mask.sum() > 0 and 'reg' in locals():
    y_reg_pred = reg.predict(X_test[placed_test_mask].values)
    print("\nSalary Metrics (for placed students):")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_reg_test[placed_test_mask], y_reg_pred)):.2f}")
    print(f"R² Score: {r2_score(y_reg_test[placed_test_mask], y_reg_pred):.4f}")

# 7. Feature Importance
print_feature_importance(clf.feature_importances_, X.columns, "Placement Feature Importance")
if 'reg' in locals():
    print_feature_importance(reg.feature_importances_, X.columns, "Salary Feature Importance")

# 8. Save models
clf.save_model('placement_model.zip')
if 'reg' in locals():
    reg.save_model('salary_model.zip')
print("\nModels saved successfully.")